# Final — FrozenLake Delivery
### REINFORCE · Multi-Stage Task · Reward Shaping
**Yogeshvar Reddy Kallam** · IST 597 Deep RL · Penn State Spring 2025

---

## Multi-Stage Delivery Task

5×5 grid with 4 sequential objectives:

```
G   T    G=Goal  T=Tree(wall)
  HH     H=Hole  K=Key  C=Chest
 H   
  HH
K   C
```

**Progress chain:** INITIAL →(K)→ HAS_KEY →(C)→ HAS_PRESENT →(G)→ DELIVERED

**Reward shaping:**  +0.2 key · +0.3 chest · +1.0 goal · −1.0 hole  
**Algorithm:** REINFORCE with 3D logit table `[position, progress, action]`

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.optim as optim
import torch.nn.functional as F, gymnasium as gym, random, matplotlib.pyplot as plt
from gymnasium import spaces

INITIAL=0; HAS_KEY=1; HAS_PRESENT=2; DELIVERED_PRESENT=3
AD={(0:(-1,0)),(1:(0,1)),(2:(1,0)),(3:(0,-1))}
AD={0:(-1,0),1:(0,1),2:(1,0),3:(0,-1)}
PERP={0:[3,1],1:[0,2],2:[3,1],3:[0,2]}

class FrozenLakeDeliveryEnv(gym.Env):
    def __init__(self):
        super().__init__(); self.desc=["G   T","  HH "," H   ","  HH ","K   C"]
        self.NR=len(self.desc); self.NC=len(self.desc[0]); self.NS=self.NR*self.NC
        self.observation_space=spaces.Tuple((spaces.Discrete(self.NS),spaces.Discrete(4)))
        self.action_space=spaces.Discrete(4); self.reset()
    def reset(self,seed=None,options=None):
        self.rng=random.Random(seed); self.state=(0,INITIAL); self.done=False; return self.state,{}
    def step(self,action):
        if self.done: raise RuntimeError("Call reset()")
        mv=self.rng.choice([action]+PERP[action]); pos,prog=self.state
        x,y=pos%self.NC,pos//self.NC; dx,dy=AD[mv]; nx,ny=x+dx,y+dy
        if not(0<=nx<self.NC and 0<=ny<self.NR): nx,ny=x,y
        c=self.desc[ny][nx]; r=0
        if c=="G" and prog==DELIVERED_PRESENT: r=1.
        elif c=="K" and prog==INITIAL: r=0.2
        elif c=="C" and prog==HAS_KEY: r=0.3
        elif c=="H": r=-1.
        if c=="H": self.done=True
        elif c=="K" and prog==INITIAL: prog=HAS_KEY
        elif c=="C" and prog==HAS_KEY: prog=HAS_PRESENT
        elif c=="G" and prog==DELIVERED_PRESENT: self.done=True
        self.state=(nx+ny*self.NC,prog); return self.state,r,self.done,False,{}

env=FrozenLakeDeliveryEnv()
SHAPE=(env.observation_space[0].n, env.observation_space[1].n); NA=env.action_space.n

class DeliveryPolicy(nn.Module):
    def __init__(self): super().__init__(); self.logits=nn.Parameter(torch.zeros(SHAPE+(NA,)))
    def forward(self,state): pos,prog=state; return F.softmax(self.logits[pos,prog],dim=-1)

pol=DeliveryPolicy(); opt=optim.Adam(pol.parameters(),lr=0.01); ep_rets=[]
print("Training REINFORCE on FrozenLakeDelivery (25,000 episodes)...")
for ep in range(25_000):
    s,_=env.reset(); lps,rs=[],[]; done=False
    while not done:
        pr=pol(s); a=np.random.choice(NA,p=pr.detach().numpy())
        ns,r,done,_,_=env.step(a); lps.append(torch.log(pr[a])); rs.append(r); s=ns
    G=loss=0.
    for t in range(len(rs)-1,-1,-1): G=rs[t]+0.99*G; loss+=-lps[t]*G
    opt.zero_grad(); loss.backward(); opt.step()
    ep_rets.append(sum(rs))
    if (ep+1)%5000==0: print(f"  Ep {ep+1:>6} | avg={np.mean(ep_rets[-5000:]):.5f}")
print("✅ Done.")
w=1000; plt.figure(figsize=(10,4))
plt.plot(np.convolve(ep_rets,np.ones(w)/w,'valid'))
plt.xlabel("Episode"); plt.ylabel("Reward"); plt.title("FrozenLake Delivery — REINFORCE")
plt.tight_layout(); plt.show()
